In [1]:
import logging
import time
from pathlib import Path
from collections import deque
from typing import Dict, Any
import numpy as np

import gymnasium as gym
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecNormalize
from stable_baselines3.common.callbacks import BaseCallback, CallbackList
import src.gymnasium_envs.convex_optimization_env

PROJECT_ROOT = Path().resolve().parent.parent

log_dir = PROJECT_ROOT / "logs"
model_dir = PROJECT_ROOT / "models"

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Using device: {device}")
if torch.cuda.is_available():
    logger.info(f"GPU: {torch.cuda.get_device_name(0)}")
    logger.info(f"CUDA Version: {torch.version.cuda}")

2026-05-27 13:41:38 [INFO] Using device: cuda
2026-05-27 13:41:38 [INFO] GPU: NVIDIA GeForce GTX 1650
2026-05-27 13:41:38 [INFO] CUDA Version: 11.8


Adding status logging

In [3]:
class StatusCallback(BaseCallback):
    def __init__(self, window_size=1000):
        super().__init__()
        self.window = deque(maxlen=window_size)
        self.steps_to_converge = deque(maxlen=window_size)

    def _on_step(self):
        infos = self.locals["infos"]
        for info in infos:
            status = info.get("status")
            if status == "converged":
                self.window.append(1)
                self.steps_to_converge.append(info["iteration"])
            elif status == "diverged":
                self.window.append(-1)

        if len(self.window) > 0:
            self.logger.record("custom/converge_rate",
                self.window.count(1) / len(self.window))
            self.logger.record("custom/diverge_rate",
                self.window.count(-1) / len(self.window))

        if len(self.steps_to_converge) > 0:
            self.logger.record("custom/mean_steps_to_converge",
                np.mean(self.steps_to_converge))  # чем меньше тем лучше

        return True

class BestConvergeCallback(BaseCallback):
    def __init__(self, vec_normalize_env, model_save_path, vec_normalize_save_path, verbose=1):
        super().__init__(verbose)
        self.vec_normalize_env = vec_normalize_env
        self.model_save_path = model_save_path
        self.vec_normalize_save_path = vec_normalize_save_path
        self.best_converge_rate = -np.inf

    def _on_step(self) -> bool:
        converge_rate = self.logger.name_to_value.get("custom/converge_rate", None)

        if converge_rate is not None and converge_rate > self.best_converge_rate:
            self.best_converge_rate = converge_rate
            self.model.save(self.model_save_path)
            self.vec_normalize_env.save(self.vec_normalize_save_path)
            if self.verbose:
                logger.info(f"New best converge_rate: {converge_rate:.4f}")
        return True

Let's create a function for training

In [4]:
def train_model(config: Dict[str, Any], log_dir: str = "logs", model_dir: str = "models"):
    in_features_name = "any" if config["in_features"] is None else config["in_features"]
    logger.info(f"Starting training for {in_features_name}D task. Timesteps: {config['timesteps']}, Envs: {config['n_envs']}")

    start_time    = time.time()
    model_path    = Path(f"{model_dir}/{in_features_name}d_convex")
    best_model_path = Path(f"{model_dir}/{in_features_name}d_convex_best")
    stats_path    = Path(f"{model_dir}/{in_features_name}d_convex_vec_normalize.pkl")

    model_path.parent.mkdir(parents=True, exist_ok=True)
    best_model_path.mkdir(parents=True, exist_ok=True)

    vec_env = None
    try:
        vec_env = make_vec_env(
            "convex_optimization_env/ConvexOptimization-v1",
            n_envs=config["n_envs"],
            env_kwargs={"in_features": config["in_features"]}
        )
        vec_env = VecNormalize(vec_env, norm_obs=False, norm_reward=True)

        best_converge_callback = BestConvergeCallback(
            vec_normalize_env=vec_env,
            model_save_path=str(best_model_path / "best_model"),
            vec_normalize_save_path=str(best_model_path / "best_vec_normalize.pkl"),
            verbose=1
        )

        model = PPO(
            "MultiInputPolicy",
            vec_env,
            verbose=1,
            tensorboard_log=f"{log_dir}/{in_features_name}d/",
            n_steps=config["n_steps"],
            n_epochs=config["n_epochs"],
            batch_size=config["batch_size"],
            learning_rate=config["learning_rate"],
            gamma=config["gamma"],
            clip_range=config["clip_range"],
            ent_coef=config["ent_coef"],
            policy_kwargs=config["policy_kwargs"]
        )

        model.learn(
            total_timesteps=config["timesteps"],
            tb_log_name=f"convex_{in_features_name}d",
            callback=CallbackList([
                StatusCallback(),
                best_converge_callback
            ])
        )

        # Сохраняем финальную модель отдельно
        model.save(str(model_path))
        vec_env.save(str(stats_path))

        elapsed_time = (time.time() - start_time) / 60
        logger.info(f"Training for {in_features_name}D completed. Duration: {elapsed_time:.2f} min.")
        logger.info(f"Final model saved to: {model_path}")
        logger.info(f"Best model saved to: {best_model_path / 'best_model'}")

    except Exception as e:
        logger.error(f"Error during training for {in_features_name}D: {str(e)}", exc_info=True)
    finally:
        if vec_env is not None:
            vec_env.close()

In [5]:
config_base = {
    "in_features"   : None,
    "timesteps"     : 5_000_000, 
    "n_envs"        : 32,
    "n_steps"       : 2048,
    "n_epochs"      : 10, 
    "batch_size"    : 256, 
    "learning_rate" : 3e-4,
    "gamma"         : 0.99,
    "clip_range"    : 0.2,
    "ent_coef"      : 0.01,
    "policy_kwargs" : {
        "net_arch": dict(pi=[64, 64], vf=[64, 64])
    }
}

Let's train a 2D model

In [6]:
config = config_base
config["in_features"] = 2

train_model(config, log_dir, model_dir)

2026-05-27 13:41:46 [INFO] Starting training for 2D task. Timesteps: 5000000, Envs: 32
c:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\venv\Lib\site-packages\gymnasium\envs\registration.py:728: UserWarning: WARN: The environment is being initialised with render_mode='rgb_array' that is not in the possible render_modes (['ansi']).
  logger.warn(


Using cuda device
Logging to C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\logs/2d/convex_2d_4


2026-05-27 13:41:48 [INFO] New best converge_rate: 0.0000
c:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\venv\Lib\site-packages\numpy\linalg\_linalg.py:2767: RuntimeWarning: overflow encountered in dot
  sqnorm = x.dot(x)
C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\src\optimization\optimization_functions\convex_function.py:87: RuntimeWarning: overflow encountered in matmul
  return float(x.T @ self._A @ x + self._b @ x + self._c)
C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\src\gymnasium_envs\convex_optimization_env\envs\convex_optimization_v1.py:314: RuntimeWarning: overflow encountered in dot
  cos_sim = np.dot(grad_unit, prev_grad_unit)
C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\src\optimization\optimization_functions\convex_function.py:92: RuntimeWarning: overflow encountered in matmul
  return 2 * self._A @ x + self._b
C:\Users\Lolik\Documents\GitHub\Reinfor

---------------------------------
| custom/            |          |
|    converge_rate   | 0        |
|    diverge_rate    | 1        |
| rollout/           |          |
|    ep_len_mean     | 592      |
|    ep_rew_mean     | -205     |
| time/              |          |
|    fps             | 5690     |
|    iterations      | 1        |
|    time_elapsed    | 11       |
|    total_timesteps | 65536    |
---------------------------------
-----------------------------------------
| custom/                 |             |
|    converge_rate        | 0           |
|    diverge_rate         | 1           |
| rollout/                |             |
|    ep_len_mean          | 662         |
|    ep_rew_mean          | -241        |
| time/                   |             |
|    fps                  | 2972        |
|    iterations           | 2           |
|    time_elapsed         | 44          |
|    total_timesteps      | 131072      |
| train/                  |             |
|    approx_

2026-05-27 13:45:39 [INFO] New best converge_rate: 0.0041


...

2026-05-27 14:01:34 [INFO] New best converge_rate: 0.9990


--------------------------------------------
| custom/                   |              |
|    converge_rate          | 0.999        |
|    diverge_rate           | 0.001        |
|    mean_steps_to_converge | 241          |
| rollout/                  |              |
|    ep_len_mean            | 238          |
|    ep_rew_mean            | 16.9         |
| time/                     |              |
|    fps                    | 2031         |
|    iterations             | 37           |
|    time_elapsed           | 1193         |
|    total_timesteps        | 2424832      |
| train/                    |              |
|    approx_kl              | 0.0035449658 |
|    clip_fraction          | 0.0288       |
|    clip_range             | 0.2          |
|    entropy_loss           | -0.21        |
|    explained_variance     | 0.463        |
|    learning_rate          | 0.0003       |
|    loss                   | -0.00312     |
|    n_updates              | 360          |
|    polic

2026-05-27 14:04:47 [INFO] New best converge_rate: 1.0000


--------------------------------------------
| custom/                   |              |
|    converge_rate          | 1            |
|    diverge_rate           | 0            |
|    mean_steps_to_converge | 175          |
| rollout/                  |              |
|    ep_len_mean            | 200          |
|    ep_rew_mean            | 15.9         |
| time/                     |              |
|    fps                    | 2031         |
|    iterations             | 43           |
|    time_elapsed           | 1387         |
|    total_timesteps        | 2818048      |
| train/                    |              |
|    approx_kl              | 0.0035020988 |
|    clip_fraction          | 0.0264       |
|    clip_range             | 0.2          |
|    entropy_loss           | -0.0108      |
|    explained_variance     | 0.519        |
|    learning_rate          | 0.0003       |
|    loss                   | -0.00208     |
|    n_updates              | 420          |
|    polic

2026-05-27 14:24:22 [INFO] Training for 2D completed. Duration: 42.60 min.
2026-05-27 14:24:22 [INFO] Final model saved to: C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\models\2d_convex
2026-05-27 14:24:22 [INFO] Best model saved to: C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\models\2d_convex_best\best_model


In [7]:
config = config_base
config["in_features"] = 5

train_model(config, log_dir, model_dir)

2026-05-27 14:24:22 [INFO] Starting training for 5D task. Timesteps: 5000000, Envs: 32


Using cuda device
Logging to C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\logs/5d/convex_5d_1


c:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\venv\Lib\site-packages\gymnasium\envs\registration.py:728: UserWarning: WARN: The environment is being initialised with render_mode='rgb_array' that is not in the possible render_modes (['ansi']).
  logger.warn(
2026-05-27 14:24:22 [INFO] New best converge_rate: 0.0000


---------------------------------
| custom/            |          |
|    converge_rate   | 0        |
|    diverge_rate    | 1        |
| rollout/           |          |
|    ep_len_mean     | 642      |
|    ep_rew_mean     | -226     |
| time/              |          |
|    fps             | 4818     |
|    iterations      | 1        |
|    time_elapsed    | 13       |
|    total_timesteps | 65536    |
---------------------------------
------------------------------------------
| custom/                 |              |
|    converge_rate        | 0            |
|    diverge_rate         | 1            |
| rollout/                |              |
|    ep_len_mean          | 642          |
|    ep_rew_mean          | -225         |
| time/                   |              |
|    fps                  | 2786         |
|    iterations           | 2            |
|    time_elapsed         | 47           |
|    total_timesteps      | 131072       |
| train/                  |              |

2026-05-27 14:36:22 [INFO] New best converge_rate: 0.0020
2026-05-27 14:36:28 [INFO] New best converge_rate: 0.0040


...

2026-05-27 15:02:14 [INFO] New best converge_rate: 0.9980
2026-05-27 15:02:19 [INFO] New best converge_rate: 0.9990


--------------------------------------------
| custom/                   |              |
|    converge_rate          | 0.999        |
|    diverge_rate           | 0.001        |
|    mean_steps_to_converge | 449          |
| rollout/                  |              |
|    ep_len_mean            | 425          |
|    ep_rew_mean            | 16.6         |
| time/                     |              |
|    fps                    | 1922         |
|    iterations             | 67           |
|    time_elapsed           | 2283         |
|    total_timesteps        | 4390912      |
| train/                    |              |
|    approx_kl              | 0.0031043268 |
|    clip_fraction          | 0.0249       |
|    clip_range             | 0.2          |
|    entropy_loss           | 0.0695       |
|    explained_variance     | 0.474        |
|    learning_rate          | 0.0003       |
|    loss                   | 0.00468      |
|    n_updates              | 660          |
|    polic

2026-05-27 15:03:33 [INFO] New best converge_rate: 1.0000


--------------------------------------------
| custom/                   |              |
|    converge_rate          | 1            |
|    diverge_rate           | 0            |
|    mean_steps_to_converge | 422          |
| rollout/                  |              |
|    ep_len_mean            | 424          |
|    ep_rew_mean            | 16.2         |
| time/                     |              |
|    fps                    | 1920         |
|    iterations             | 69           |
|    time_elapsed           | 2354         |
|    total_timesteps        | 4521984      |
| train/                    |              |
|    approx_kl              | 0.0033848898 |
|    clip_fraction          | 0.0342       |
|    clip_range             | 0.2          |
|    entropy_loss           | 0.0995       |
|    explained_variance     | 0.477        |
|    learning_rate          | 0.0003       |
|    loss                   | 0.00112      |
|    n_updates              | 680          |
|    polic

2026-05-27 15:08:38 [INFO] Training for 5D completed. Duration: 44.26 min.
2026-05-27 15:08:38 [INFO] Final model saved to: C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\models\5d_convex
2026-05-27 15:08:38 [INFO] Best model saved to: C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\models\5d_convex_best\best_model


In [8]:
config = config_base
config["in_features"] = 10

train_model(config, log_dir, model_dir)

2026-05-27 15:08:38 [INFO] Starting training for 10D task. Timesteps: 5000000, Envs: 32


Using cuda device
Logging to C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\logs/10d/convex_10d_1


2026-05-27 15:08:38 [INFO] New best converge_rate: 0.0000


---------------------------------
| custom/            |          |
|    converge_rate   | 0        |
|    diverge_rate    | 1        |
| rollout/           |          |
|    ep_len_mean     | 632      |
|    ep_rew_mean     | -220     |
| time/              |          |
|    fps             | 4865     |
|    iterations      | 1        |
|    time_elapsed    | 13       |
|    total_timesteps | 65536    |
---------------------------------
-----------------------------------------
| custom/                 |             |
|    converge_rate        | 0           |
|    diverge_rate         | 1           |
| rollout/                |             |
|    ep_len_mean          | 662         |
|    ep_rew_mean          | -232        |
| time/                   |             |
|    fps                  | 2774        |
|    iterations           | 2           |
|    time_elapsed         | 47          |
|    total_timesteps      | 131072      |
| train/                  |             |
|    approx_

2026-05-27 15:14:56 [INFO] New best converge_rate: 0.0027


...

2026-05-27 15:52:16 [INFO] New best converge_rate: 0.2137
2026-05-27 15:52:17 [INFO] New best converge_rate: 0.2145
2026-05-27 15:52:19 [INFO] New best converge_rate: 0.2154
2026-05-27 15:52:20 [INFO] New best converge_rate: 0.2162
2026-05-27 15:52:24 [INFO] New best converge_rate: 0.2170
2026-05-27 15:52:28 [INFO] New best converge_rate: 0.2179


--------------------------------------------
| custom/                   |              |
|    converge_rate          | 0.218        |
|    diverge_rate           | 0.782        |
|    mean_steps_to_converge | 766          |
| rollout/                  |              |
|    ep_len_mean            | 985          |
|    ep_rew_mean            | 7.87         |
| time/                     |              |
|    fps                    | 1892         |
|    iterations             | 76           |
|    time_elapsed           | 2632         |
|    total_timesteps        | 4980736      |
| train/                    |              |
|    approx_kl              | 0.0039536036 |
|    clip_fraction          | 0.0282       |
|    clip_range             | 0.2          |
|    entropy_loss           | 0.233        |
|    explained_variance     | 0.593        |
|    learning_rate          | 0.0003       |
|    loss                   | -0.00558     |
|    n_updates              | 750          |
|    polic

2026-05-27 15:52:53 [INFO] New best converge_rate: 0.2187
2026-05-27 15:52:53 [INFO] New best converge_rate: 0.2195
2026-05-27 15:52:56 [INFO] New best converge_rate: 0.2203
2026-05-27 15:53:01 [INFO] New best converge_rate: 0.2212
2026-05-27 15:53:04 [INFO] New best converge_rate: 0.2220


--------------------------------------------
| custom/                   |              |
|    converge_rate          | 0.222        |
|    diverge_rate           | 0.778        |
|    mean_steps_to_converge | 769          |
| rollout/                  |              |
|    ep_len_mean            | 992          |
|    ep_rew_mean            | 8.07         |
| time/                     |              |
|    fps                    | 1891         |
|    iterations             | 77           |
|    time_elapsed           | 2668         |
|    total_timesteps        | 5046272      |
| train/                    |              |
|    approx_kl              | 0.0038212352 |
|    clip_fraction          | 0.0306       |
|    clip_range             | 0.2          |
|    entropy_loss           | 0.247        |
|    explained_variance     | 0.687        |
|    learning_rate          | 0.0003       |
|    loss                   | 0.00281      |
|    n_updates              | 760          |
|    polic

2026-05-27 15:53:26 [INFO] Training for 10D completed. Duration: 44.81 min.
2026-05-27 15:53:26 [INFO] Final model saved to: C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\models\10d_convex
2026-05-27 15:53:26 [INFO] Best model saved to: C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\models\10d_convex_best\best_model


In [9]:
config = config_base
config["in_features"] = 100

train_model(config, log_dir, model_dir)

2026-05-27 15:53:26 [INFO] Starting training for 100D task. Timesteps: 5000000, Envs: 32


Using cuda device
Logging to C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\logs/100d/convex_100d_1


2026-05-27 15:53:27 [INFO] New best converge_rate: 0.0000


---------------------------------
| custom/            |          |
|    converge_rate   | 0        |
|    diverge_rate    | 1        |
| rollout/           |          |
|    ep_len_mean     | 642      |
|    ep_rew_mean     | -224     |
| time/              |          |
|    fps             | 4625     |
|    iterations      | 1        |
|    time_elapsed    | 14       |
|    total_timesteps | 65536    |
---------------------------------
------------------------------------------
| custom/                 |              |
|    converge_rate        | 0            |
|    diverge_rate         | 1            |
| rollout/                |              |
|    ep_len_mean          | 661          |
|    ep_rew_mean          | -232         |
| time/                   |              |
|    fps                  | 2596         |
|    iterations           | 2            |
|    time_elapsed         | 50           |
|    total_timesteps      | 131072       |
| train/                  |              |

2026-05-27 16:38:48 [INFO] Training for 100D completed. Duration: 45.36 min.
2026-05-27 16:38:48 [INFO] Final model saved to: C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\models\100d_convex
2026-05-27 16:38:48 [INFO] Best model saved to: C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\models\100d_convex_best\best_model


In [10]:
config = config_base
config["in_features"] = None

train_model(config, log_dir, model_dir)

2026-05-27 16:38:48 [INFO] Starting training for anyD task. Timesteps: 5000000, Envs: 32


Using cuda device
Logging to C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\logs/anyd/convex_anyd_1


2026-05-27 16:38:48 [INFO] New best converge_rate: 0.0000


---------------------------------
| custom/            |          |
|    converge_rate   | 0        |
|    diverge_rate    | 1        |
| rollout/           |          |
|    ep_len_mean     | 622      |
|    ep_rew_mean     | -221     |
| time/              |          |
|    fps             | 4740     |
|    iterations      | 1        |
|    time_elapsed    | 13       |
|    total_timesteps | 65536    |
---------------------------------
------------------------------------------
| custom/                 |              |
|    converge_rate        | 0            |
|    diverge_rate         | 1            |
| rollout/                |              |
|    ep_len_mean          | 622          |
|    ep_rew_mean          | -209         |
| time/                   |              |
|    fps                  | 2753         |
|    iterations           | 2            |
|    time_elapsed         | 47           |
|    total_timesteps      | 131072       |
| train/                  |              |

2026-05-27 16:51:02 [INFO] New best converge_rate: 0.0021


...

2026-05-27 17:22:41 [INFO] New best converge_rate: 0.3009
2026-05-27 17:22:41 [INFO] New best converge_rate: 0.3017
2026-05-27 17:22:42 [INFO] New best converge_rate: 0.3026
2026-05-27 17:22:43 [INFO] New best converge_rate: 0.3034
2026-05-27 17:22:44 [INFO] New best converge_rate: 0.3042
2026-05-27 17:22:46 [INFO] New best converge_rate: 0.3051
2026-05-27 17:22:48 [INFO] New best converge_rate: 0.3059


-------------------------------------------
| custom/                   |             |
|    converge_rate          | 0.306       |
|    diverge_rate           | 0.694       |
|    mean_steps_to_converge | 429         |
| rollout/                  |             |
|    ep_len_mean            | 947         |
|    ep_rew_mean            | 8.26        |
| time/                     |             |
|    fps                    | 1885        |
|    iterations             | 76          |
|    time_elapsed           | 2642        |
|    total_timesteps        | 4980736     |
| train/                    |             |
|    approx_kl              | 0.003745273 |
|    clip_fraction          | 0.0337      |
|    clip_range             | 0.2         |
|    entropy_loss           | 0.443       |
|    explained_variance     | 0.673       |
|    learning_rate          | 0.0003      |
|    loss                   | 0.00229     |
|    n_updates              | 750         |
|    policy_gradient_loss   | -0

2026-05-27 17:23:11 [INFO] New best converge_rate: 0.3068
2026-05-27 17:23:18 [INFO] New best converge_rate: 0.3076
2026-05-27 17:23:18 [INFO] New best converge_rate: 0.3084
2026-05-27 17:23:19 [INFO] New best converge_rate: 0.3093
2026-05-27 17:23:25 [INFO] New best converge_rate: 0.3101


-------------------------------------------
| custom/                   |             |
|    converge_rate          | 0.31        |
|    diverge_rate           | 0.69        |
|    mean_steps_to_converge | 427         |
| rollout/                  |             |
|    ep_len_mean            | 948         |
|    ep_rew_mean            | 7.65        |
| time/                     |             |
|    fps                    | 1884        |
|    iterations             | 77          |
|    time_elapsed           | 2678        |
|    total_timesteps        | 5046272     |
| train/                    |             |
|    approx_kl              | 0.003944257 |
|    clip_fraction          | 0.038       |
|    clip_range             | 0.2         |
|    entropy_loss           | 0.462       |
|    explained_variance     | 0.629       |
|    learning_rate          | 0.0003      |
|    loss                   | 0.00837     |
|    n_updates              | 760         |
|    policy_gradient_loss   | -0

2026-05-27 17:23:47 [INFO] Training for anyD completed. Duration: 44.98 min.
2026-05-27 17:23:47 [INFO] Final model saved to: C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\models\anyd_convex
2026-05-27 17:23:47 [INFO] Best model saved to: C:\Users\Lolik\Documents\GitHub\Reinforcement-learning-for-Gradient-descent\models\anyd_convex_best\best_model
